In [1]:
# ~~ TERCEROS ~~ #
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# ~~ ESTE PROYECTO ~~ #
import set_paths
from config.paths import *
from src.preprocessing import concat_tables
from src.mkt import (embudo, add_unique_interested)
from src.utils import get_files

# EMBUDOS
### Exposición -> Visualizaciones -> Consultas recibidas

## TOTAL MKT

In [35]:
#test = concat_tables(ESTAD_PORTALES)
test_mkt = concat_tables(dir= ESTAD_PORTALES,
                     start= 2,
                     stop = 4,
                     output= None)

test_mkt = test_mkt.groupby(by = "Período").sum()

inter_df = concat_tables(dir = INTERESADOS,
                           start = 2,
                           stop = 4,
                           output= None)

add_unique_interested(estad_port_df= test_mkt,
                      inter_df= inter_df,
                      inplace= True)


embudo_test = embudo(raw_data= test_mkt, 
                     columns= ["Exposición", "Visualizaciones", "Consultas recibidas", "Interesados"],
                     pct_decimals= 5)
embudo_test

,conteo,pct_tot,pct_rel
Exposición,220521,1.00000,1.00000
Visualizaciones,7732,0.03506,0.03506
Consultas recibidas,234,0.00106,0.03026
Interesados,228,0.00103,0.97436


## POR PORTAL

In [4]:
estads_portales = get_files(ESTAD_PORTALES)
inter_portales = get_files(INTERESADOS)

for i, portal in enumerate(estads_portales):

    test = pd.read_excel(ESTAD_PORTALES / portal)

    inter_test = pd.read_excel(INTERESADOS / inter_portales[i])

    add_unique_interested(estad_port_df= test,
                          inter_df= inter_test,
                          inplace= True)
    
    embudo_test = embudo(raw_data= test, 
                         columns= ["Exposición", "Visualizaciones", "Consultas recibidas", "Interesados"],
                         pct_decimals= 5)
    
    print(f" ~~ PORTAL {i+1} ~~ ")
    print(embudo_test.head())
    print()

 ~~ PORTAL 1 ~~ 
                     conteo  pct_tot  pct_rel
Exposición           126028  1.00000  1.00000
Visualizaciones        5450  0.04324  0.04324
Consultas recibidas     131  0.00104  0.02404
Interesados             127  0.00101  0.96947

 ~~ PORTAL 2 ~~ 
                     conteo  pct_tot  pct_rel
Exposición           131415  1.00000  1.00000
Visualizaciones        4449  0.03385  0.03385
Consultas recibidas     117  0.00089  0.02630
Interesados             112  0.00085  0.95726

 ~~ PORTAL 3 ~~ 
                     conteo  pct_tot  pct_rel
Exposición            35000  1.00000  1.00000
Visualizaciones        1784  0.05097  0.05097
Consultas recibidas      87  0.00249  0.04877
Interesados              86  0.00246  0.98851

 ~~ PORTAL 4 ~~ 
                     conteo  pct_tot  pct_rel
Exposición            54106  1.00000  1.00000
Visualizaciones        1499  0.02770  0.02770
Consultas recibidas      30  0.00055  0.02001
Interesados              30  0.00055  1.00000



In [39]:
def join_inter_crm(crm_df: pd.DataFrame,
                   inter_df: pd.DataFrame) -> pd.DataFrame:


    # CONCILIAR LAS TABLAS
    crm_df.rename(columns = {"Email":"E-mail", "ID de publicación": "Id aviso"}, inplace= True)
    crm_df = crm_df[crm_df["Portal origen"] == "Inmuebles24"].copy()
    s = pd.to_numeric(crm_df["Id aviso"], errors="coerce")

    crm_df["Id aviso"] = s.where(s.mod(1).eq(0), 0).fillna(0).astype(int)
    inter_df["Id aviso"] = (
    pd.to_numeric(inter_df["Id aviso"], errors="coerce")
    .fillna(0)
    .astype(int)
    )

    # INNER JOIN

    resultado = inter_df[["E-mail", "Sucursal", "Id aviso"]].merge(
    crm_df[["E-mail", "¿Fue traspasado?", "Código asesor", "Id aviso"]],
    on=["E-mail", "Id aviso"],
    how="inner"
)



    return resultado

In [40]:
crm_df = pd.read_csv(CRM / get_files(CRM)[0])
join_inter_crm(crm_df= crm_df, inter_df= inter_df)

,E-mail,Sucursal,Id aviso,¿Fue traspasado?,Código asesor
0,stephafig305@gmail.com,Altaltium Real Estates Premium,149937387,Sí,Lesly Madariaga
1,danhuerta8@gmail.com,Altaltium Real Estates Premium,149976162,Sí,Lesly Madariaga
2,alejandra.abdrngl@gmail.com,Altaltium Real Estates Premium,146639224,Sí,Lesly Madariaga
3,albeeto.garcia.ortiz@gmail.com,Altaltium Real Estates Premium,150322088,Sí,Lesly Madariaga
4,euromchino@gmail.com,Altaltium Real Estates Premium,149700647,Sí,Lesly Madariaga
...,...,...,...,...,...
218,eaplannermx@gmail.com,Ventas 4,148674908,Sí,Alfredo Jimenez
219,toscanojoselin5@gmail.com,Ventas 4,149963789,Sí,Alfredo Jimenez
220,nalle1990.ramlop@gmail.com,Ventas 4,149963899,Sí,Alfredo Jimenez
221,israel.espinosa@fovissste.gob.mx,Ventas 4,148628039,Sí,Lesly Madariaga


In [37]:
crm_df = pd.read_csv(CRM / get_files(CRM)[0])
crm_df.rename(columns = {"Email":"E-mail", "ID de publicación": "Id aviso"}, inplace= True)
crm_df = crm_df[crm_df["Portal origen"] == "Inmuebles24"]
s = pd.to_numeric(crm_df["Id aviso"], errors="coerce")

crm_df["Id aviso"] = s.where(s.mod(1).eq(0), 0).fillna(0).astype(int)
#crm_df["Id aviso"] = crm_df["Id aviso"].astype(int)
inter_df["Id aviso"] = (
    pd.to_numeric(inter_df["Id aviso"], errors="coerce")
    .fillna(0)
    .astype(int)
)
#interesados = inter_df


In [38]:
#INNER JOIN SIN DUPLICADOS (MÁS COMO UNA INTERSECCIÓN DE CONJUNTOS)
resultado = inter_df[["E-mail", "Sucursal", "Id aviso"]].merge(
    crm_df[["E-mail", "¿Fue traspasado?", "Código asesor", "Id aviso"]],
    on=["E-mail", "Id aviso"],
    how="inner"
)#.drop_duplicates() #TOMA LEADS QUE ESTÁN LAS TABLAS DE INTERESADOS DE IM24 Y REGISTRO CRM
                    #SOLO LOS QUE ESTÁN EN AMBAS TABLAS
